# 03. 변수 생성·선택 (v2 — 신규 외부 변수 포함)

`integrated.csv` 위에 다음 변수군을 만든다.

1. **기존 가공 변수 (11종)**: avg_ticket·momentum·anchor_score 등
2. **신규 외부 변수 (10종)**: 임대료·공실률·인구밀도·1인가구 등
3. **신규 파생 변수 (6종)**: rent_to_sales·single_ratio 등
4. **타깃**: log_sales (회귀) + sales_class 3분위 (분류)
5. **인코딩**: 자치구·업종·분기 원-핫

산출물: `data/processed/features.csv` + `data_dictionary.md`

In [1]:
# 표준 라이브러리 + 강의 범위만
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)

PROCESSED_DIR = Path('/Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project/data/processed')
df = pd.read_csv(PROCESSED_DIR / 'integrated.csv')        # 02의 산출물 입력
print('integrated:', df.shape, '컬럼:', df.shape[1])
df.head(3)

integrated: (22099, 83) 컬럼: 83


,gu,biz,q,ML_SELNG_AMT,ML_SELNG_CO,WKEND_SELNG_CO,TMZON_06_11_SELNG_AMT,TMZON_17_21_SELNG_CO,AGRDE_60_ABOVE_SELNG_AMT,TMZON_11_14_SELNG_CO,WKEND_SELNG_AMT,SUN_SELNG_AMT,AGRDE_10_SELNG_AMT,AGRDE_30_SELNG_AMT,FML_SELNG_CO,TUES_SELNG_AMT,AGRDE_10_SELNG_CO,THUR_SELNG_AMT,TMZON_21_24_SELNG_CO,MON_SELNG_CO,THSMON_SELNG_CO,AGRDE_40_SELNG_AMT,TMZON_14_17_SELNG_CO,TMZON_00_06_SELNG_AMT,TMZON_21_24_SELNG_AMT,FRI_SELNG_AMT,FRI_SELNG_CO,MON_SELNG_AMT,AGRDE_60_ABOVE_SELNG_CO,TMZON_00_06_SELNG_CO,AGRDE_50_SELNG_AMT,AGRDE_20_SELNG_CO,FML_SELNG_AMT,MDWK_SELNG_AMT,TMZON_17_21_SELNG_AMT,AGRDE_50_SELNG_CO,MDWK_SELNG_CO,THSMON_SELNG_AMT,SAT_SELNG_CO,SAT_SELNG_AMT,...,TUES_SELNG_CO,SUN_SELNG_CO,TMZON_06_11_SELNG_CO,AGRDE_30_SELNG_CO,AGRDE_20_SELNG_AMT,TMZON_11_14_SELNG_AMT,THUR_SELNG_CO,WED_SELNG_CO,STOR_CO,OPBIZ_RT,CLSBIZ_RT,TOT_FLPOP_CO,ML_FLPOP_CO,FML_FLPOP_CO,TMZON_11_14_FLPOP_CO,TMZON_17_21_FLPOP_CO,SAT_FLPOP_CO,SUN_FLPOP_CO,OPR_SALE_MT_AVRG,CHG_DOMINANT,TOT_WRC_POPLTN_CO,SUBWAY_STATN_CO,BUS_STTN_CO,UNIV_CO,GNRL_HSPTL_CO,PBLOFC_CO,event_count,rent_index_q,rent_growth_yoy,vacancy_rate,transaction_count_q,avg_transaction_price,margin_proxy,gu_population,gu_pop_density,gu_foreign,q_int,year,closure_count,single_household
0,강남구,PC방,20201,641937821.0,122457.0,50943.0,39522157.0,48291.0,16993134.0,14385.0,266077914.0,125011376.0,51488997.0,139983724.0,34320.0,104711608.0,12791.0,103529482.0,29858.0,20765.0,156777.0,42581138.0,29441.0,137853936.0,159208404.0,114716671.0,23094.0,111211048.0,2523.0,26311.0,73159767.0,95824.0,162158449.0,538018356.0,250224617.0,12597.0,105834.0,804096270.0,27102.0,141066538.0,...,20399.0,23841.0,8491.0,25661.0,479889510.0,75040514.0,20869.0,20707.0,2.0,50.0,50.0,97146119.0,47805406.0,49340712.0,15722303.0,17886920.0,11714029.0,10726758.0,87.718447,다이나믹,9826.222857,0.210938,3.8125,0.085938,0.273438,0.789062,0,NaN,NaN,NaN,NaN,NaN,NaN,539231.0,13651.4177,4824.0,20201,2020,5215.0,NaN
1,강남구,PC방,20202,607436880.0,100605.0,41560.0,33146968.0,40270.0,18697862.0,12528.0,245172130.0,116193824.0,43557049.0,108427902.0,18321.0,87736054.0,11085.0,87173480.0,24299.0,15292.0,118926.0,46490388.0,21075.0,95200991.0,138135985.0,98571047.0,16646.0,85880149.0,3310.0,15065.0,20019360.0,74581.0,87821454.0,450086204.0,230900852.0,3614.0,77366.0,695258334.0,21300.0,128978306.0,...,14745.0,20260.0,5689.0,17476.0,458065773.0,79838524.0,15125.0,15558.0,7.0,0.0,0.0,95019033.0,46753092.0,48265941.0,15252081.0,17427733.0,11706705.0,10561185.0,88.815534,다이나믹,9826.222857,0.210938,3.8125,0.085938,0.273438,0.789062,0,NaN,NaN,NaN,NaN,NaN,NaN,539231.0,13651.4177,4824.0,20202,2020,5215.0,NaN
2,강남구,PC방,20203,160755265.0,35768.0,12892.0,15486939.0,11137.0,4898622.0,5593.0,57200492.0,28363510.0,24962209.0,23563611.0,9182.0,27385405.0,6741.0,26886536.0,9746.0,7066.0,44950.0,7151757.0,8150.0,29121367.0,38714410.0,31614306.0,6951.0,31181407.0,1192.0,7813.0,21953415.0,27795.0,41594787.0,145149560.0,56775445.0,4410.0,32058.0,202350052.0,6332.0,28836982.0,...,5941.0,6560.0,2511.0,3454.0,119820438.0,24941243.0,5664.0,6436.0,4.0,0.0,0.0,94185725.0,46344898.0,47840831.0,15102301.0,17179633.0,11658457.0,10504199.0,88.679612,다이나믹,9826.222857,0.210938,3.8125,0.085938,0.273438,0.789062,0,NaN,NaN,NaN,NaN,NaN,NaN,539231.0,13651.4177,4824.0,20203,2020,5215.0,NaN


## 1. 기본 가공 변수 11종 — 각 변수의 가설/이유 주석

In [2]:
# (1) avg_ticket = 매출 ÷ 건수 → 객단가 (작은 가게 다수 vs 큰 가게 소수 구분)
df['avg_ticket'] = df['THSMON_SELNG_AMT'] / df['THSMON_SELNG_CO'].replace(0, np.nan)

# (2) momentum = 직전 분기 대비 매출 변화 (자치구·업종 시계열 — 성장 모멘텀)
df = df.sort_values(['gu','biz','q_int'])                  # diff 전 정렬 필수
df['momentum'] = df.groupby(['gu','biz'])['THSMON_SELNG_AMT'].diff() / 1e6   # 백만 단위로 스케일

# (3) anchor_score = 지하철×2 + 대학×3 + 병원×2 + 공공기관×1 (대학 비중 가장 큼 — 청년 유동)
df['anchor_score'] = (df['SUBWAY_STATN_CO']*2 + df['UNIV_CO']*3 +
                       df['GNRL_HSPTL_CO']*2 + df['PBLOFC_CO']*1)

# (4) lunch_bias = 11~14시 매출 / 전체 매출 (점심 장사 비중 — 오피스 상권 시그널)
df['lunch_bias'] = df['TMZON_11_14_SELNG_AMT'] / df['THSMON_SELNG_AMT'].replace(0, np.nan)

# (5) weekend_bias = 주말 매출 / 전체 (관광/주거 vs 평일 오피스 구분)
df['weekend_bias'] = df['WKEND_SELNG_AMT'] / df['THSMON_SELNG_AMT'].replace(0, np.nan)

# (6) age_entropy = 연령 매출 분포 엔트로피 (특정 연령 집중도 vs 다양성)
age_cols = [c for c in df.columns if c.startswith('AGRDE_') and c.endswith('_SELNG_AMT')]
age_mat = df[age_cols].clip(lower=0)                       # 음수 방지
row_sum = age_mat.sum(axis=1).replace(0, np.nan)
age_ratio = age_mat.div(row_sum, axis=0).fillna(0)         # 각 연령대 비율
df['age_entropy'] = -(age_ratio * np.log(age_ratio.replace(0, 1))).sum(axis=1)  # H = -Σp*log(p)

# (7) peer_avg_sales = 동일 분기·업종 평균 매출 (해당 (gu,biz)가 평균 대비 어디인가)
df['peer_avg_sales'] = df.groupby(['biz','q_int'])['THSMON_SELNG_AMT'].transform('mean')

# (8) closure_density = 폐업률 (그대로 — 변환 없음)
df['closure_density'] = df['CLSBIZ_RT'].fillna(0)

# (9) covid_phase: 0=정상, 1=충격(20Q1~21Q4), 2=회복(22Q1~)
df['covid_phase'] = 0
df.loc[df['q_int'].between(20201, 20214), 'covid_phase'] = 1
df.loc[df['q_int'].between(20221, 20264), 'covid_phase'] = 2

# (10) competitor_density = 점포수 / 유동인구 (포화도)
df['competitor_density'] = df['STOR_CO'] / df['TOT_FLPOP_CO'].replace(0, np.nan)

# (11) season_amplitude = 분기 매출 변동성 (표준편차/평균 — 계절성 강한 업종 식별)
seasonal = (df.groupby(['gu','biz'])['THSMON_SELNG_AMT']
              .agg(lambda s: s.std()/max(s.mean(), 1))
              .rename('season_amplitude').reset_index())
df = df.merge(seasonal, on=['gu','biz'], how='left')

base_features = ['avg_ticket','momentum','anchor_score','lunch_bias','weekend_bias',
                 'age_entropy','peer_avg_sales','closure_density','covid_phase',
                 'competitor_density','season_amplitude']
print('생성된 기본 변수:', base_features)
df[base_features].describe().T

생성된 기본 변수: ['avg_ticket', 'momentum', 'anchor_score', 'lunch_bias', 'weekend_bias', 'age_entropy', 'peer_avg_sales', 'closure_density', 'covid_phase', 'competitor_density', 'season_amplitude']


,count,mean,std,min,25%,50%,75%,max
avg_ticket,22099.0,1.173365e+05,3.102029e+05,9.985405e+01,1.768027e+04,3.855438e+04,1.210135e+05,1.805714e+07
momentum,20861.0,8.568021e+01,2.432569e+04,-1.025185e+06,-3.745812e+02,2.827760e-01,4.171709e+02,1.027375e+06
anchor_score,22099.0,1.514036e+00,5.605131e-01,8.488372e-01,1.028986e+00,1.410714e+00,1.835714e+00,3.237569e+00
lunch_bias,22099.0,2.373315e-01,1.537914e-01,0.000000e+00,1.383794e-01,2.341078e-01,3.133202e-01,1.000000e+00
weekend_bias,22099.0,2.289281e-01,1.401277e-01,0.000000e+00,1.294106e-01,2.376357e-01,3.177372e-01,1.000000e+00
age_entropy,22099.0,1.324121e+00,3.525771e-01,-0.000000e+00,1.214134e+00,1.454724e+00,1.566974e+00,1.751069e+00
peer_avg_sales,22099.0,3.403123e+09,6.504871e+09,5.358215e+07,6.567956e+08,1.508156e+09,3.625004e+09,8.194775e+10
closure_density,22099.0,5.118512e+00,1.657011e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.000000e+02
covid_phase,22099.0,1.661749e+00,4.731250e-01,1.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00
competitor_density,17919.0,3.503864e-07,2.220977e-06,0.000000e+00,4.991411e-08,1.192675e-07,2.882054e-07,2.218217e-04


## 2. 신규 파생 변수 (외부 데이터 활용)

In [3]:
# (a) rent_to_sales — 임대료지수 ÷ log(매출) → 임대료 부담 지표 (낮을수록 매출이 임대료 잘 흡수)
df['rent_to_sales'] = df['rent_index_q'] / np.log1p(df['THSMON_SELNG_AMT'])

# (b) vacancy_change_q — 분기 공실률 변화 (자치구 시계열) → 상권 분위기 변화 시그널
df = df.sort_values(['gu','q_int'])
df['vacancy_change_q'] = df.groupby('gu')['vacancy_rate'].diff()

# (c) pop_density_log — 인구밀도 로그 (분포가 long-tail이라 log 변환)
df['pop_density_log'] = np.log1p(df['gu_pop_density'])

# (d) single_ratio — 1인가구 / 자치구 인구 (1인가구 비중이 큰 곳일수록 외식·편의점 친화적)
df['single_ratio'] = df['single_household'] / df['gu_population'].replace(0, np.nan)

# (e) closure_per_store — 폐업 / 점포 (자치구의 사업 안정성 지표)
df['closure_per_store'] = df['closure_count'] / df['STOR_CO'].replace(0, np.nan)

# (f) margin_zscore — 마진 proxy의 z-score (전체 분포 기준 표준화)
mu, sd = df['margin_proxy'].mean(), df['margin_proxy'].std()
df['margin_zscore'] = (df['margin_proxy'] - mu) / sd if sd > 0 else 0

# (g) gu_foreign_ratio — 외국인 비율 (관광·다국적 상권 시그널)
df['gu_foreign_ratio'] = df['gu_foreign'] / df['gu_population'].replace(0, np.nan)

new_external = ['rent_index_q','rent_growth_yoy','vacancy_rate','margin_proxy',
                'gu_pop_density','closure_count','single_household',
                'transaction_count_q','avg_transaction_price','gu_foreign_ratio']
new_derived = ['rent_to_sales','vacancy_change_q','pop_density_log',
               'single_ratio','closure_per_store','margin_zscore']
print('신규 외부 변수:', new_external)
print('신규 파생 변수:', new_derived)
df[new_external + new_derived].describe().T

신규 외부 변수: ['rent_index_q', 'rent_growth_yoy', 'vacancy_rate', 'margin_proxy', 'gu_pop_density', 'closure_count', 'single_household', 'transaction_count_q', 'avg_transaction_price', 'gu_foreign_ratio']
신규 파생 변수: ['rent_to_sales', 'vacancy_change_q', 'pop_density_log', 'single_ratio', 'closure_per_store', 'margin_zscore']


,count,mean,std,min,25%,50%,75%,max
rent_index_q,10750.0,9.999307e+01,1.424709e+00,9.530000e+01,9.949854e+01,1.000000e+02,1.004139e+02,1.093025e+02
rent_growth_yoy,10622.0,7.436811e-01,1.220226e+00,-3.184312e+00,4.944444e-02,4.931850e-01,1.130500e+00,6.895000e+00
vacancy_rate,10718.0,7.317741e+00,3.025923e+00,1.750000e+00,4.800000e+00,7.200000e+00,9.373611e+00,1.566667e+01
margin_proxy,1861.0,3.046318e+03,7.563597e+02,1.560393e+03,2.505387e+03,3.052884e+03,3.520269e+03,4.491426e+03
gu_pop_density,22099.0,1.668528e+04,4.554910e+03,6.247762e+03,1.400060e+04,1.674197e+04,1.972129e+04,2.609138e+04
closure_count,14843.0,2.803306e+03,1.196806e+03,1.164000e+03,1.951000e+03,2.540000e+03,3.321000e+03,7.250000e+03
single_household,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_count_q,10908.0,1.029686e+02,7.167515e+01,1.400000e+01,5.200000e+01,8.400000e+01,1.370000e+02,6.290000e+02
avg_transaction_price,10908.0,2.416007e+09,2.535705e+09,2.506212e+08,8.086096e+08,1.467538e+09,2.599527e+09,1.279124e+10
gu_foreign_ratio,22099.0,3.016539e-02,2.417925e-02,6.468731e-03,8.946073e-03,2.367180e-02,4.135201e-02,7.501597e-02


## 3. 타깃 — 자치구 매출 + **점포당 매출** (단일 점포 예측)

자치구 전체 매출은 입지의 잠재력 지표지만 단일 점포 평가에는 부적합(자치구당 점포 수가 다름).
→ **`sales_per_store = THSMON_SELNG_AMT / STOR_CO`** 를 새 타깃으로 추가.

| 타깃 | 단위 | 용도 |
|---|---|---|
| `log_sales` | 자치구·업종·분기 합산 매출 | 04/05/06 자치구 단위 분석 |
| `log_sales_per_store` | **단일 점포 추정 월매출** | **07/09 — 사용자 입지 평가** |
| `sales_class` / `store_class` | 3분위 (low/mid/high) | 분류 모델 |

In [4]:
# 매출 0/음수, 점포수 0 행은 log·나눗셈 불가 → 제거
df = df[(df['THSMON_SELNG_AMT'] > 0) & (df['STOR_CO'].fillna(0) > 0)].copy()

# (1) 자치구 합산 매출 로그 (기존 타깃)
df['log_sales'] = np.log(df['THSMON_SELNG_AMT'])

# (2) 점포당 매출 — 단일 점포 매출의 추정치
#   자치구 전체 매출 ÷ 그 자치구·업종 분기의 점포수
#   ex) 강남구 한식 1500점포가 750억 → 점포당 ≈ 500만원/월
df['sales_per_store']     = df['THSMON_SELNG_AMT'] / df['STOR_CO']
df['log_sales_per_store'] = np.log(df['sales_per_store'])

# 3분위 경계 — 자치구 매출 기준
q33,  q66  = df['log_sales'].quantile([0.33, 0.66])
df['sales_class'] = pd.cut(df['log_sales'],
                            bins=[-np.inf, q33, q66, np.inf],
                            labels=['low','mid','high']).astype(str)

# 3분위 경계 — 점포당 매출 기준 (단일 점포 분류 타깃)
sq33, sq66 = df['log_sales_per_store'].quantile([0.33, 0.66])
df['store_class'] = pd.cut(df['log_sales_per_store'],
                            bins=[-np.inf, sq33, sq66, np.inf],
                            labels=['low','mid','high']).astype(str)

print('자치구 매출 분위 분포:'); print(df['sales_class'].value_counts())
print('\n점포당 매출 분위 분포:'); print(df['store_class'].value_counts())
print('\n점포당 매출 (월) 분포:')
print((df['sales_per_store']/1e6).describe().round(2), '(백만원)')

자치구 매출 분위 분포:
sales_class
high    5973
low     5798
mid     5797
Name: count, dtype: int64

점포당 매출 분위 분포:
store_class
high    5973
low     5798
mid     5797
Name: count, dtype: int64

점포당 매출 (월) 분포:
count     17568.00
mean        588.30
std        4953.85
min           0.01
25%          23.99
50%          91.93
75%         320.80
max      335792.96
Name: sales_per_store, dtype: float64 (백만원)


## 4. 원-핫 인코딩

In [5]:
# 트리·선형 모델 모두 카테고리 변수는 원-핫 형태가 필요
df['quarter_num'] = df['q'].astype(str).str[-1].astype(int)              # 분기 1/2/3/4
df_oh  = pd.get_dummies(df['quarter_num'].astype(int), prefix='Q')        # Q_1, Q_2, Q_3, Q_4
df_gu  = pd.get_dummies(df['gu'], prefix='gu')                            # 25개 자치구
df_biz = pd.get_dummies(df['biz'], prefix='biz')                          # 10개 업종
df = pd.concat([df, df_oh, df_gu, df_biz], axis=1)
print('인코딩 후 shape:', df.shape)

인코딩 후 shape: (17568, 186)


## 5. 변수 선택 — 분산 0 + 상관 |r|≥0.9 필터

In [6]:
# 모델 입력 후보 = raw 핵심 + 가공 + 신규 외부 + 신규 파생 + 원-핫
raw_core = ['STOR_CO','OPBIZ_RT','CLSBIZ_RT','TOT_FLPOP_CO','TOT_WRC_POPLTN_CO',
            'SUBWAY_STATN_CO','BUS_STTN_CO','UNIV_CO','GNRL_HSPTL_CO','PBLOFC_CO',
            'OPR_SALE_MT_AVRG','event_count']
cat_oh = [c for c in df.columns if c.startswith(('Q_','gu_','biz_'))]
feature_cols = raw_core + base_features + new_external + new_derived + cat_oh
feature_cols = [c for c in feature_cols if c in df.columns]
print('초기 변수 수:', len(feature_cols))

# (1) 분산 0 제거 — 모든 행이 같으면 학습에 무의미
num_only = [c for c in feature_cols if df[c].dtype != object]
varzero = [c for c in num_only if df[c].nunique(dropna=False) <= 1]
print('분산 0 제거:', varzero)
feature_cols = [c for c in feature_cols if c not in varzero]

# (2) 상관 |r|>=0.9 — 두 변수 거의 같은 정보 → 다중공선성 + 모델 복잡도만 증가
numeric_subset = [c for c in feature_cols if not c.startswith(('Q_','gu_','biz_'))]
corr = df[numeric_subset].corr().abs().copy()
arr = corr.values.copy()                                  # numpy array 만들기 (read-only 회피)
np.fill_diagonal(arr, 0)                                  # 자기상관(1)은 제외
corr = pd.DataFrame(arr, index=corr.index, columns=corr.columns)
drop = set()
for i, ccol in enumerate(corr.columns):
    if ccol in drop: continue
    # ccol과 상관 0.9 이상인 모든 변수 후보 → drop 후보로 추가 (ccol은 살림)
    high = corr[ccol][corr[ccol] >= 0.9].index.tolist()
    for h in high:
        if h != ccol and h not in drop:
            drop.add(h)
print('상관 |r|>=0.9 제거:', sorted(drop))
feature_cols = [c for c in feature_cols if c not in drop]
print('최종 입력 변수 수:', len(feature_cols))

초기 변수 수: 122
분산 0 제거: ['single_household', 'single_ratio']
상관 |r|>=0.9 제거: ['closure_density', 'competitor_density', 'margin_zscore']
최종 입력 변수 수: 117


## 6. features.csv 저장

In [7]:
id_cols = ['gu','biz','q','q_int']
# 타깃 6종 — 자치구 매출 3종 + 점포당 매출 3종
target_cols = ['THSMON_SELNG_AMT','log_sales','sales_class',
                'sales_per_store','log_sales_per_store','store_class']
out_cols = id_cols + target_cols + feature_cols
out_path = PROCESSED_DIR / 'features.csv'
df[out_cols].to_csv(out_path, index=False)
print('저장:', out_path, '|', df[out_cols].shape)

저장: /Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project/data/processed/features.csv | (17568, 127)


## 7. 데이터 사전 저장

In [8]:
# 보고서 §4.2 표로 직접 인용할 수 있는 사전
dict_rows = [
    ('THSMON_SELNG_AMT','월매출(원)','타깃'),
    ('log_sales','log(매출)','타깃 회귀'),
    ('sales_class','매출 3분위','타깃 분류'),
    ('avg_ticket','매출÷건수','기존 가공'),
    ('momentum','전분기 대비 매출 차','기존 가공'),
    ('anchor_score','지하철·대학·병원·공공기관 가중합','기존 가공'),
    ('lunch_bias','점심 매출 비중','기존 가공'),
    ('weekend_bias','주말 매출 비중','기존 가공'),
    ('age_entropy','연령 매출분포 엔트로피','기존 가공'),
    ('peer_avg_sales','동일분기 업종 평균 매출','기존 가공'),
    ('closure_density','폐업률','기존 가공'),
    ('covid_phase','코로나 시기 0/1/2','기존 가공'),
    ('competitor_density','점포수÷유동인구','기존 가공'),
    ('season_amplitude','계절 변동성','기존 가공'),
    ('rent_index_q','임대료지수','신규 외부 (team1)'),
    ('rent_growth_yoy','임대료 yoy','신규 외부 (team1)'),
    ('vacancy_rate','공실률','신규 외부 (team1)'),
    ('margin_proxy','마진 proxy','신규 외부 (team1)'),
    ('transaction_count_q','실거래 건수','신규 외부 (team1)'),
    ('avg_transaction_price','실거래 평균가','신규 외부 (team1)'),
    ('gu_pop_density','자치구 인구밀도','신규 외부 (team4)'),
    ('closure_count','연 폐업점포수','신규 외부 (team4)'),
    ('single_household','1인가구','신규 외부 (team4)'),
    ('gu_foreign_ratio','외국인 비율','신규 파생'),
    ('rent_to_sales','임대료지수÷log매출','신규 파생'),
    ('vacancy_change_q','공실률 분기 변화','신규 파생'),
    ('pop_density_log','log(인구밀도)','신규 파생'),
    ('single_ratio','1인가구÷인구','신규 파생'),
    ('closure_per_store','폐업÷점포','신규 파생'),
    ('margin_zscore','마진 z-score','신규 파생'),
]
dd = pd.DataFrame(dict_rows, columns=['변수','설명','구분'])
# 마크다운 파일로도 저장 → 보고서에 그대로 붙여쓰기
dict_md = ['# 데이터 사전 — features.csv (v2)\n',
           '| 변수 | 설명 | 구분 |\n|---|---|---|']
for _, r in dd.iterrows():
    dict_md.append(f'| `{r["변수"]}` | {r["설명"]} | {r["구분"]} |')
(PROCESSED_DIR / 'data_dictionary.md').write_text('\n'.join(dict_md), encoding='utf-8')
dd

,변수,설명,구분
0,THSMON_SELNG_AMT,월매출(원),타깃
1,log_sales,log(매출),타깃 회귀
2,sales_class,매출 3분위,타깃 분류
3,avg_ticket,매출÷건수,기존 가공
4,momentum,전분기 대비 매출 차,기존 가공
5,anchor_score,지하철·대학·병원·공공기관 가중합,기존 가공
6,lunch_bias,점심 매출 비중,기존 가공
7,weekend_bias,주말 매출 비중,기존 가공
8,age_entropy,연령 매출분포 엔트로피,기존 가공
9,peer_avg_sales,동일분기 업종 평균 매출,기존 가공


## 다음 단계
- 04_models: 회귀·분류 7종 + 데이터 변환 비교
- 05_interpret: 잔차·중요도 해석
- 06_map: 자치구 지도
- 07_startup_evaluator: 입지 평가기